# 12 — Agentic RAG (Tool-Using AI)

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Understand what an *AI agent* is in practical terms.
2. Wire up four tools: documents, tables, graph, calculator.
3. Run a transparent ReAct-style loop and read the trace.
4. Apply safe-agent patterns.


## What is an agent?

Plain RAG is **one-shot**: retrieve → answer. An **agent** is *iterative*: the LLM is given a list of tools (functions) it can call, and it decides *which* tool to call next based on what it has seen so far. We see every step.

Tools in this notebook:
1. `search_documents(query)` — RAG over PDFs.
2. `query_table(name, expr)` — pandas-style query over a known CSV/XLSX.
3. `query_graph(question)` — canned graph queries.
4. `calculate(expr)` — safe arithmetic.


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
import pandas as pd
from pathlib import Path
from src.rag_utils import build_store_from_folder, rag_answer
from src.graph_utils import build_finance_graph
from src.agentic_rag_utils import safe_calc, make_table_tool, make_graph_tool, run_agent

# 1) Build the RAG store and the knowledge graph
store = build_store_from_folder('data/generated/pdf')
G = build_finance_graph()

# 2) Load CSV/XLSX into a tables dictionary
tables = {
    'sales':       pd.read_csv('data/generated/csv/01_sales_transactions.csv'),
    'purchases':   pd.read_csv('data/generated/csv/02_purchase_transactions.csv'),
    'journals':    pd.read_csv('data/generated/csv/03_journal_entries.csv'),
    'vendors':     pd.read_csv('data/generated/csv/04_vendor_master.csv'),
    'rpt':         pd.read_excel('data/generated/xlsx/10_related_party_transactions.xlsx'),
    'ar_aging':    pd.read_excel('data/generated/xlsx/04_accounts_receivable_aging.xlsx'),
}
for name, df in tables.items():
    print(f'  {name:12s} {df.shape}')

## 12.1 — Wire up the four tools

In [ ]:
def search_documents(query):
    answer, hits = rag_answer(query, store, k=4, return_hits=True)
    sources = ', '.join(f"{h['metadata'].get('source')}#p{h['metadata'].get('page','-')}" for h in hits)
    return f'{answer}\n[sources: {sources}]'

tools = {
    'search_documents': search_documents,
    'query_table':      make_table_tool(tables),
    'query_graph':      make_graph_tool(G),
    'calculate':        lambda x: str(safe_calc(x)),
}
print('Tools:', list(tools))

## 12.2 — Run the agent on a CA-style task

In [ ]:
task = (
    'Identify all related-party purchases above NPR 5,00,000 from the purchases table, '
    'check which vendor is a related party (use the graph), and confirm whether each was '
    'approved per the procurement policy (use the documents). Report findings with citations.'
)
trace = run_agent(task, tools=tools, max_steps=6, verbose=True)
print('\n=== FINAL ANSWER ===')
print(trace.answer)

## 12.3 — Inspect the trace

In [ ]:
for s in trace.steps:
    print(f'-- step {s["step"]} -- action={s.get("action")}')
    print('  input:', (s.get('input') or '')[:200])
    if 'observation' in s:
        print('  observation:', s['observation'][:200])
    print()

## Expected output

The agent typically performs 3-5 actions: one `query_table` for purchases over a threshold, one `query_graph` for related parties, one `search_documents` to find the approval rule. On a mock LLM the trace shows the pattern but the answer is canned.

## Exercise

1. Run the agent on: *"Estimate the year-end inventory provision if we doubled the provisioning rate on >365-day items."* (uses table + calculator)
2. Lower `max_steps` to 2 and see how the answer degrades.
3. Replace `search_documents` with a no-op and see what the agent does instead.


## Common errors

| Symptom | Fix |
|---|---|
| Agent never says FINAL | Lower temperature; verify the system prompt is loaded; raise `max_steps`. |
| Wrong table syntax | `query_table` expects `name | expression`; e.g. `purchases | amount > 500000`. |
| Mock LLM answer | Set a real key — agents are noticeably better with stronger models. |


## ⚠️ Professional caution

Agents have **more autonomy** than plain RAG. The minimum safety hygiene:
1. **Bound the loop** (`max_steps`).
2. **Safe tools only** — no `exec`, no shell, no internet writes.
3. **Log every step** (we do — see the trace) so a reviewer can audit what happened.
4. **Never let the agent take real-world actions** (send email, post payments) without a human in the loop.